# The Price Is Right - Week 7

## Finetuning Opensource LLM (meta-llama/Llama-3.2-3B)

If you are using LITE_MODE=True, then please run this on a free T4 box.

If you are using LITE_MODE-False, then please use a paid A100 with high memory.

## Overview

This notebook walks through the end-to-end process of fine-tuning **Meta's Llama 3.2 3B** model to predict product prices — a regression task framed as causal language modeling.

**Key techniques used:**
- **QLoRA (Quantized Low-Rank Adaptation):** The base model is loaded in 4-bit precision using `bitsandbytes`, drastically reducing memory usage. LoRA adapters are then trained on top, keeping the base weights frozen.
- **SFTTrainer from TRL:** Hugging Face's `trl` library provides `SFTTrainer` for supervised fine-tuning, handling tokenization, padding, and training loop orchestration.
- **Weights & Biases:** Experiment metrics (loss, learning rate, etc.) are logged to W&B for monitoring and comparison across runs.
- **Hugging Face Hub:** Checkpoints are pushed to the Hub periodically, enabling easy resumption if the runtime is interrupted.

**Workflow:**
1. Install dependencies and import libraries
2. Configure hyperparameters (LoRA rank, learning rate, batch size, etc.)
3. Authenticate with Hugging Face and Weights & Biases
4. Load the dataset of product descriptions with prices
5. Load the quantized base model and tokenizer
6. Configure LoRA adapters and training arguments
7. Train and push the fine-tuned model to the Hub

## 1. Environment Setup

Install the required packages:
- **bitsandbytes** — enables 4-bit and 8-bit quantization for memory-efficient model loading
- **trl** — provides `SFTTrainer` for supervised fine-tuning with LoRA/QLoRA support

We also fetch `util.py`, a helper module containing the `Tester` class for evaluating model predictions with visualizations.

In [ ]:
!pip install -q --upgrade bitsandbytes==0.48.2 trl==0.25.1
!wget -q https://raw.githubusercontent.com/patrickcmd/llm_engineering/main/week7/util.py -O util.py

### Import Libraries

Core libraries used throughout this notebook:

| Library | Purpose |
|---------|---------|
| `transformers` | Model loading, tokenization, and training configuration |
| `datasets` | Loading and managing HuggingFace datasets |
| `peft` | Parameter-Efficient Fine-Tuning (LoRA configuration) |
| `trl` | `SFTTrainer` and `SFTConfig` for supervised fine-tuning |
| `bitsandbytes` | Quantization support (loaded implicitly via `BitsAndBytesConfig`) |
| `wandb` | Experiment tracking and metric logging |
| `torch` | PyTorch backend for model computation |

In [ ]:
import os
import re
import math
from tqdm import tqdm
# from google.colab import userdata
from huggingface_hub import login
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, set_seed, BitsAndBytesConfig
from datasets import load_dataset, Dataset, DatasetDict
import wandb
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datetime import datetime
import matplotlib.pyplot as plt
from getpass import getpass

### Check GPU Compute Capability

We query the CUDA compute capability to determine whether the GPU supports **bfloat16** (bf16) precision. GPUs with compute capability **>= 8.0** (e.g., A100, H100) natively support bf16, which offers better numerical stability than fp16 during training. Older GPUs like the T4 (capability 7.5) fall back to fp16.

In [ ]:
capability = torch.cuda.get_device_capability()
capability

## 2. Configuration & Hyperparameters

All configurable parameters are defined upfront. The notebook supports two modes controlled by `LITE_MODE`:

| Parameter | LITE_MODE=True (T4) | LITE_MODE=False (A100) |
|-----------|---------------------|------------------------|
| Epochs | 1 | 3 |
| Batch size | 32 | 256 |
| LoRA rank (r) | 32 | 256 |
| Target modules | Attention only | Attention + MLP |
| Dataset | `items_prompts_lite` | `items_prompts_full` |

**QLoRA Hyperparameters:**
- **`LORA_R`** — Rank of the low-rank decomposition. Higher rank = more trainable parameters = better expressiveness but more memory.
- **`LORA_ALPHA`** — Scaling factor, set to `2 × LORA_R` as a common heuristic.
- **`TARGET_MODULES`** — Which linear layers in the transformer to apply LoRA adapters to. In full mode, we target both attention projections and MLP layers.
- **`QUANT_4_BIT`** — When True, the base model is loaded in NF4 (4-bit NormalFloat) quantization with double quantization for further memory savings.

**Training Hyperparameters:**
- **Cosine learning rate schedule** with a brief warmup (`WARMUP_RATIO=0.01`) to stabilize early training.
- **`paged_adamw_32bit`** optimizer — a memory-efficient variant of AdamW that pages optimizer states to CPU when GPU memory is tight.

In [ ]:
# Constants

BASE_MODEL = "meta-llama/Llama-3.2-3B"
PROJECT_NAME = "price"
HF_USER = "patrickcmd" # your HF name here!

LITE_MODE = False

DATA_USER = "patrickcmd"
DATASET_NAME = f"{DATA_USER}/items_prompts_lite" if LITE_MODE else f"{DATA_USER}/items_prompts_full"

RUN_NAME =  f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
if LITE_MODE:
  RUN_NAME += "-lite"
PROJECT_RUN_NAME = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME = f"{HF_USER}/{PROJECT_RUN_NAME}"

# Hyper-parameters - overall

EPOCHS = 1 if LITE_MODE else 3
BATCH_SIZE = 32 if LITE_MODE else 256
MAX_SEQUENCE_LENGTH = 128
GRADIENT_ACCUMULATION_STEPS = 1

# Hyper-parameters - QLoRA

QUANT_4_BIT = True
LORA_R = 32 if LITE_MODE else 256
LORA_ALPHA = LORA_R * 2
ATTENTION_LAYERS = ["q_proj", "v_proj", "k_proj", "o_proj"]
MLP_LAYERS = ["gate_proj", "up_proj", "down_proj"]
TARGET_MODULES = ATTENTION_LAYERS if LITE_MODE else ATTENTION_LAYERS + MLP_LAYERS
LORA_DROPOUT = 0.1

# Hyper-parameters - training

LEARNING_RATE = 1e-4
WARMUP_RATIO = 0.01
LR_SCHEDULER_TYPE = 'cosine'
WEIGHT_DECAY = 0.001
OPTIMIZER = "paged_adamw_32bit"

capability = torch.cuda.get_device_capability()
use_bf16 = capability[0] >= 8

# Tracking

VAL_SIZE = 500 if LITE_MODE else 1000
LOG_STEPS = 5 if LITE_MODE else 10
SAVE_STEPS = 100 if LITE_MODE else 200
LOG_TO_WANDB = True

In [ ]:
# A100 GPU supports this; T4 does not natively

use_bf16

### A Note on Optimizers

We use `paged_adamw_32bit` — a variant of [AdamW](https://huggingface.co/docs/transformers/main/en/perf_train_gpu_one#optimizers) that pages optimizer states to CPU memory when GPU memory pressure is high. This is particularly important with QLoRA because the base model weights, quantization states, and optimizer states all compete for GPU memory.

Standard Adam/AdamW stores two additional state tensors (first and second moment estimates) per trainable parameter. With a LoRA rank of 256 across many layers, these states can consume significant memory — paged AdamW mitigates this by offloading to CPU on demand.

## 3. Authentication

### Log in to HuggingFace and Weights & Biases

You need API tokens from both services:

1. **HuggingFace** — sign up at [huggingface.co](https://huggingface.co), create a token with write access, and add it as a secret called `HF_TOKEN`.
2. **Weights & Biases** — sign up at [wandb.ai](https://wandb.ai), copy your API key, and add it as a secret called `WANDB_API_KEY`.

If running in Google Colab, add these under the Secrets panel (key icon in the left sidebar). The cells below authenticate interactively via `login()` and `getpass()`.

In [ ]:
# Log in to HuggingFace

# hf_token = userdata.get('HF_TOKEN')
# login(hf_token, add_to_git_credential=True)
login()

In [ ]:
# Log in to Weights & Biases
# wandb_api_key = userdata.get('WANDB_API_KEY')
wandb_api_key = getpass('WANDB_API_KEY: ')
os.environ["WANDB_API_KEY"] = wandb_api_key
wandb.login()

# Configure Weights & Biases to record against our project
os.environ["WANDB_PROJECT"] = PROJECT_NAME
os.environ["WANDB_LOG_MODEL"] = "false"
os.environ["WANDB_WATCH"] = "false"

## 4. Load and Prepare the Dataset

The dataset is hosted on the Hugging Face Hub and contains three splits: **train**, **val**, and **test**. Each example has a `prompt` field (product description formatted as a text prompt) and a `completion` field (the target price).

- In **full mode**, the training set contains 800,000 examples with 1,000 validation samples.
- In **lite mode**, a smaller dataset variant is used with 500 validation samples.

The validation set is truncated to `VAL_SIZE` for faster evaluation during training.

In [ ]:
dataset = load_dataset(DATASET_NAME)
train = dataset['train']
val = dataset['val'].select(range(VAL_SIZE))
test = dataset['test']

In [ ]:
# if you wish to reduce the training dataset to 10,000 points instead, then uncomment this line:

# train = train.select(range(10000))

### Initialize the W&B Run

Create a new Weights & Biases run to track training metrics. The run name includes a timestamp so each experiment is uniquely identifiable. W&B will log training loss, evaluation loss, learning rate, and other metrics at each `LOG_STEPS` interval.

In [ ]:
if LOG_TO_WANDB:
  wandb.init(
      # Set the wandb entity where your project will be logged (generally your team name).
    entity="patrickcmd",
    project=PROJECT_NAME, 
    name=RUN_NAME
  )

## 5. Load the Tokenizer and Model

The base model is loaded with **quantization** — reducing weight precision from 16 bits to 4 bits (NF4 format). This cuts memory usage by ~4x while preserving most of the model's capability. Double quantization (`bnb_4bit_use_double_quant`) further compresses the quantization constants themselves.

The tokenizer is configured with right-side padding and the EOS token is reused as the pad token, which is standard practice for causal language models.

In [ ]:
# pick the right quantization

if QUANT_4_BIT:
  quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    bnb_4bit_quant_type="nf4"
  )
else:
  quant_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
  )

In [ ]:
# Load the Tokenizer and the Model

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id

print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:.1f} MB")

## 6. Configure Training

Two configuration objects are required:

1. **`LoraConfig`** — defines the LoRA adapter architecture: which layers to target, the rank of the low-rank matrices, dropout rate, and the scaling factor (alpha). Setting `bias="none"` means bias terms are not trained, and `task_type="CAUSAL_LM"` ensures compatibility with autoregressive generation.

2. **`SFTConfig`** — wraps all training arguments: batch size, learning rate schedule, checkpointing strategy, Hub integration, and evaluation settings. Key choices include:
   - **`push_to_hub=True`** with **`hub_strategy="every_save"`** — uploads checkpoints every `SAVE_STEPS` steps, providing resilience against runtime interruptions.
   - **`max_length=128`** — truncates sequences to 128 tokens, which is sufficient for price prediction prompts and keeps training efficient.
   - **`eval_strategy="steps"`** — runs evaluation on the validation set at each save checkpoint.

In [ ]:
# LoRA Parameters

lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

In [ ]:
# Training parameters

train_parameters = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=10,
    logging_steps=LOG_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    fp16=not use_bf16,
    bf16=use_bf16,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=WARMUP_RATIO,
    # group_by_length=True,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to="wandb" if LOG_TO_WANDB else None,
    run_name=RUN_NAME,
    max_length=MAX_SEQUENCE_LENGTH,
    save_strategy="steps",
    hub_strategy="every_save",
    push_to_hub=True,
    hub_model_id=HUB_MODEL_NAME,
    hub_private_repo=True,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS
)

### Create the SFTTrainer

`SFTTrainer` combines the base model, LoRA config, training arguments, and datasets into a single trainer object. During initialization, it automatically appends EOS tokens to each example, tokenizes the text, and truncates sequences to `max_length`.

In [ ]:
fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train,
    eval_dataset=val,
    peft_config=lora_parameters,
    args=train_parameters
)

## 7. Fine-Tune the Model

Training will run for the configured number of epochs, saving checkpoints to the Hugging Face Hub every `SAVE_STEPS` steps.

**Runtime considerations:**
- On a free Colab T4, Google may reclaim the GPU at any time. On paid A100 plans, sessions can last up to 24 hours.
- Because checkpoints are pushed to the Hub automatically, you can resume from the last saved checkpoint if the runtime is interrupted.
- To resume training, load the model with `is_trainable=True` and call `trainer.train(resume_from_checkpoint=True)`. See [this Colab](https://colab.research.google.com/drive/1qGTDVIas_Vwoby4UVi2vwsU0tHXy8OMO#scrollTo=R_O04fKxMMT-) for a complete resume example.

After training completes, the final adapter weights are pushed to the Hub as a private repository.

In [ ]:
# Fine-tune!
fine_tuning.train()

# Push our fine-tuned model to Hugging Face
fine_tuning.model.push_to_hub(PROJECT_RUN_NAME, private=True)
print(f"Saved to the hub: {PROJECT_RUN_NAME}")

## 8. Cleanup

Finish the Weights & Biases run to flush all pending logs and finalize the experiment dashboard. After this cell, you can view the complete training metrics at [wandb.ai](https://wandb.ai).

In [ ]:
if LOG_TO_WANDB:
  wandb.finish()

## Next Steps

With the fine-tuned model pushed to the Hugging Face Hub, you can:

1. **Evaluate** — Load the adapter with `PeftModel.from_pretrained()` and run inference on the test set using the `Tester` class from `util.py`.
2. **Resume training** — If the run was interrupted, load from the latest checkpoint and continue training.
3. **Merge adapters** — Use `model.merge_and_unload()` to merge LoRA weights into the base model for faster inference without the PEFT overhead.
4. **Compare runs** — Review training curves and hyperparameter sweeps in the Weights & Biases dashboard.